# 1123 57일차

## 1. Agent

모델이 도구를 고르고 실행하고 그 결과를 다시 모델에 넣는 과정을 끝날 때까지 반복하는 것 (`f35_agent1`, `f35_agent2`)

앞서는 `tool_calls`를 꺼내 코드가 `invoke`로 돌리고, 그 결과를 다시 질문에 붙이는 것까지 직접 적어야 했음. 계산이 여러 단계면 그 왕복을 전부 손으로 이어야 함<br/>
agent는 그 반복을 대신 돌림

1. `from langchain.agents import create_agent` : agent를 만드는 함수
2. `model=llm` : 쓸 모델
3. `tools=[add, multiply]` : 등록할 도구 목록
4. `system_prompt` : 역할과 행동 지침
5. `agent.invoke({"messages": [{"role": "user", "content": "33과 7을 곱한 후 4를 더해줘."}]})` : 질문을 메시지 목록으로 넣음
6. `result["messages"][-1].content` : 최종 답

- **`bind_tools`는 도구 목록만 붙이고, agent는 실행과 결과 재투입까지 함**
- 입력이 문자열이 아니라 `{"messages": [...]}` 딕셔너리
- 메시지 하나는 `role`(보낸 쪽)과 `content`(내용)로 적음
- 돌려받는 것도 딕셔너리라 `result["messages"]`로 주고받은 전체를 꺼냄
- agent는 한 번 만들어 두고 질문마다 `invoke`만 다시 부름

## 2. 메시지 흐름

agent가 한 번 도는 동안 `result["messages"]`에 쌓이는 메시지의 종류와 순서 (`f35_agent2`, `f36_agent_tool`)

최종 답만 보면 도구를 몇 개 어떤 값으로 썼는지 알 수 없음. 메시지를 순서대로 보면 어디서 도구를 골랐고 무엇이 돌아왔는지 드러남

1. `HumanMessage` : 내가 보낸 질문
2. `AIMessage` : 모델의 응답. 도구를 골랐으면 `tool_calls`가 채워지고 `content`는 빔
3. `ToolMessage` : 도구를 실행한 결과. 호출 하나당 하나
4. `AIMessage` : 도구 결과를 받아 만든 최종 답
5. `type(msg).__name__` : 메시지 종류를 꺼냄
6. `isinstance(msg, AIMessage) and msg.tool_calls` : 도구를 고른 메시지만 거름

- **`AIMessage`는 도구를 고른 자리와 최종 답 자리에 모두 나와서, `tool_calls`가 있는지로 가름**
- 메시지 개수는 고정이 아니라 도구를 몇 번 썼는지에 따라 달라짐
- `ToolMessage`의 `content`에는 함수가 돌려준 값이 문자열로 들어감
- `enumerate`로 돌리면 몇 번째 메시지인지 같이 찍을 수 있음

#### cf) 한 번에 몰아 내는 도구 호출

- 여러 단계 계산을 시켜도 `AIMessage` 하나에 `tool_calls`가 여러 개 들어올 수 있음
- 이때 모델이 중간 결과를 미리 계산해서 다음 호출의 인자에 넣음 (`multiply(33, 7)` 뒤에 바로 `add(231, 4)`)
- 도구 결과를 받아 다음 호출을 정하는 순서가 아니라, 계획을 한 번에 적어 내는 것
- 그래서 `ToolMessage`가 연달아 쌓이고 모델을 부르는 횟수는 줄어듦

## 3. system_prompt

agent에게 역할과 지켜야 할 행동을 미리 적어 주는 문장 (`f36_agent_tool`, `f38_agent_error`, `f39_agent_project`)

도구를 등록해도 쓸지 말지는 모델이 정함. 직접 계산해 버리거나 도중에 거절할 수 있어서, 어떻게 행동할지 글로 지정함

1. `system_prompt="계산이 필요하면 도구를 사용하세요."` : 한 줄로 주는 경우
2. 괄호 안에 문자열을 나란히 두면 여러 줄로 나눠 적을 수 있음

- 역할(`당신은 계산 전문 Agent입니다`), 도구를 쓸 조건, 여러 단계일 때의 순서, 오류일 때 할 일처럼 항목을 나눠 적음
- 한국어로 답하라고 적지 않으면 수식 기호나 영어가 섞인 답이 나올 수 있음

#### cf) 괄호 안 문자열 이어 붙이기

- 괄호 안에 문자열 리터럴을 나란히 두면 파이썬이 하나로 이어 붙임
- 사이에 쉼표를 넣으면 문자열이 아니라 튜플이 됨
- 메시지 `content`에 튜플이 들어가면 블록 목록으로 바뀌어, 서버가 형식이 아니라며 400을 돌려줌
- 줄 끝에 공백을 두지 않으면 단어가 붙어서 이어짐

## 4. config

agent를 돌릴 때 같이 넘기는 실행 옵션 (`f37_config`)

agent는 멈출 조건에 닿을 때까지 반복하는 구조라, 모델이 도구를 계속 부르면 끝나지 않음. 반복 횟수에 상한을 둠

1. `agent.invoke({...}, config={"recursion_limit": 20})` : 그래프를 밟을 단계 수 상한

- **`recursion_limit`은 메시지 개수가 아니라 그래프 단계 수**
- 상한에 닿으면 답을 만들지 못하고 `GraphRecursionError`
- 모델 호출과 도구 실행이 각각 한 단계라, 도구를 한 번 쓰는 질문도 네 단계가 필요함
- 단계가 남아 있어도 최종 답이 나오면 거기서 끝남

## 5. 도구 실행 중 오류

도구로 쓴 함수가 예외를 내서 agent가 멈추는 경우 (`f38_agent_error`)

도구는 그냥 파이썬 함수라서 0으로 나누기 같은 입력에는 예외를 냄. 예외가 나면 agent가 답을 만들지 못하고 멈춤

1. `try:` 안에 `agent.invoke`를 둠
2. `except ZeroDivisionError` : 도구가 낸 예외를 종류별로 받음
3. `except Exception` : 위에서 걸리지 않은 나머지를 받음

- **도구가 낸 예외는 agent 안에서 처리되지 않고 `invoke`를 부른 자리까지 올라옴**
- `a / b`에서 `b`가 0이면 `ZeroDivisionError`
- `except`는 적어 둔 종류와 그 하위만 잡으므로, 어떤 예외가 나는지 맞춰 적어야 함
- 모델이 도구를 부르기 전에 거절하면 예외가 아예 나지 않음. 그때는 `tool_calls`가 빈 응답만 돌아옴

## 6. 반복 입력

질문을 코드에 적어 두지 않고 입력받아 계속 돌리는 구조 (`f39_agent_project`)

질문을 바꿀 때마다 파일을 고치면 모델 연결과 도구 등록도 매번 다시 함. 모델과 agent는 한 번만 만들고, 반복문에서는 질문만 받아 넘김

1. `def run_agent(question):` : agent 호출과 출력 정리를 함수로 묶음
2. `while True:` : 종료 입력이 올 때까지 돌림
3. `input("계산할 질문을 입력하세요(종료: q): ")` : 질문을 받음
4. `if question.lower() == 'q': break` : 종료
5. `if not question.strip(): continue` : 빈 입력은 넘김
6. `try` / `except`로 감싸 오류가 나도 반복문이 끊기지 않게 함

- `lower()`를 거치면 `q`와 `Q`를 같이 받음
- `strip()`이 빈 문자열이면 공백만 넣은 입력도 걸러짐
- 호출끼리는 서로 독립이라 앞 질문을 기억하지 않음. 이어서 묻게 하려면 이전 메시지를 `messages`에 같이 넣어야 함